# 🚢 Titanic: Machine Learning Solution Notebook (100% Pure ML - No Cheating)
> **Objective**: Predict survival on the Titanic using an end-to-end Machine Learning pipeline featuring Exploratory Data Analysis (EDA), domain-specific Feature Engineering (Woman-Child Group survival signals), multi-model cross-validation, and soft-voting/stacking ensembling.

---
### 📌 Pure ML Pipeline Architecture:
1. **Exploratory Data Analysis (EDA)**: Missing data analysis, feature distributions, survival correlations.
2. **Feature Engineering**: Title extraction, age imputation by Title/Pclass, Fare-per-person, Cabin decks, Family size, Sex-Pclass interactions, and Out-of-Fold **Woman-Child Group (WCG)** target encoding.
3. **Model Suite**: 8 Diverse Classifiers (CatBoost, LightGBM, XGBoost, Random Forest, Extra Trees, HistGradientBoosting, Support Vector Machine, Logistic Regression).
4. **10-Fold Stratified Cross-Validation**: 10-Fold Stratified CV to prevent leakage and compute robust Out-Of-Fold (OOF) scores.
5. **Ensembling & Stacking**: Weighted blending and meta-learner stacking (see section 6 for out-of-fold scores).
6. **Submission**: Export pure ML predictions to `output/submission.csv`.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.size'] = 11

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

print("All libraries imported successfully!")

## 1. 📂 Data Loading & Inspection

In [ ]:
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape:  {test_df.shape}")

print("\n--- Train Data Missing Values ---")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

print("\n--- Test Data Missing Values ---")
print(test_df.isnull().sum()[test_df.isnull().sum() > 0])

train_df.head()


## 2. 📊 Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Survival by Sex
sns.barplot(data=train_df, x='Sex', y='Survived', ax=axes[0, 0], palette='pastel')
axes[0, 0].set_title('Survival Rate by Sex')
axes[0, 0].set_ylim(0, 1)

# Survival by Pclass
sns.barplot(data=train_df, x='Pclass', y='Survived', ax=axes[0, 1], palette='pastel')
axes[0, 1].set_title('Survival Rate by Ticket Class (Pclass)')
axes[0, 1].set_ylim(0, 1)

# Age Distribution by Survival
sns.kdeplot(data=train_df, x='Age', hue='Survived', common_norm=False, fill=True, ax=axes[1, 0], palette='husl')
axes[1, 0].set_title('Age Distribution by Survival Status')

# Survival by Embarked Port
sns.barplot(data=train_df, x='Embarked', y='Survived', ax=axes[1, 1], palette='pastel')
axes[1, 1].set_title('Survival Rate by Embarked Port')

plt.tight_layout()
plt.show()


## 3. 🛠️ Advanced Feature Engineering & Data Preprocessing
Key engineered features:
- **Title Extraction**: Title extracted from `Name` and categorized (`Mr`, `Mrs`, `Miss`, `Master`, `Officer`, `Royalty`).
- **Age Imputation**: Missing ages imputed using the median of matching `(Title, Pclass)` groups.
- **Family Features**: `FamilySize = SibSp + Parch + 1`, `IsAlone`, and `FamilyType`.
- **Ticket Group & Fare Per Person**: Shared tickets split the fare among group members (`FarePerPerson`).
- **Cabin Deck**: Extract deck code (`A`-`G`, `U` for Unknown).
- **Sex x Pclass Interaction**: Interaction category combining `Sex` and `Pclass`.
- **Woman-Child Group (WCG) Survival Signal**: Out-of-Fold target encoding for female/child ticket groups (computed strictly on training set folds to avoid leakage).


In [ ]:
train_len = len(train_df)
df = pd.concat([train_df, test_df], ignore_index=True)

# 1. Extract Title
df['Title'] = df['Name'].apply(lambda name: re.search(r' ([A-Za-z]+)\.', name).group(1) if re.search(r' ([A-Za-z]+)\.', name) else '')

title_mapping = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Officer', 'Rev': 'Officer', 'Col': 'Officer', 'Major': 'Officer',
    'Mlle': 'Miss', 'Mme': 'Mrs', 'Ms': 'Miss', 'Lady': 'Royalty',
    'Sir': 'Royalty', 'Capt': 'Officer', 'Countess': 'Royalty',
    'Jonkheer': 'Royalty', 'Don': 'Royalty', 'Dona': 'Royalty'
}
df['Title'] = df['Title'].map(title_mapping).fillna('Rare')

# Imputation statistics come from the TRAIN rows only, so nothing about the test set
# leaks into the values the models are trained on.
train_part = df.iloc[:train_len]

# 2. Impute missing Age by Title & Pclass
age_medians = train_part.groupby(['Title', 'Pclass'])['Age'].median()
df_age_keys = pd.MultiIndex.from_frame(df[['Title', 'Pclass']])
df['Age'] = df['Age'].fillna(pd.Series(age_medians.reindex(df_age_keys).values, index=df.index))
df['Age'] = df['Age'].fillna(train_part['Age'].median())

# 3. Impute missing Fare by Pclass median
df['Fare'] = df['Fare'].fillna(df['Pclass'].map(train_part.groupby('Pclass')['Fare'].median()))

# 4. Impute missing Embarked with mode
df['Embarked'] = df['Embarked'].fillna(train_part['Embarked'].mode()[0])

# 5. Family Features
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

def get_family_type(size):
    if size == 1: return 'Alone'
    elif 2 <= size <= 4: return 'Small'
    else: return 'Large'

df['FamilyType'] = df['FamilySize'].apply(get_family_type)

# 6. Ticket Frequency & Fare Per Person
# Counted over train+test on purpose: this uses test *features* (not labels), which are
# genuinely available at prediction time, so it is transductive rather than leakage.
ticket_counts = df['Ticket'].value_counts()
df['TicketGroupSize'] = df['Ticket'].map(ticket_counts)
df['FarePerPerson'] = df['Fare'] / df['TicketGroupSize']
df['LogFare'] = np.log1p(df['Fare'])
df['LogFarePerPerson'] = np.log1p(df['FarePerPerson'])

# 7. Cabin Deck & Interactions
df['CabinDeck'] = df['Cabin'].apply(lambda x: str(x)[0] if pd.notna(x) else 'U')
df['HasCabin'] = (df['Cabin'].notna()).astype(int)
df['IsChild'] = ((df['Age'] < 12) | (df['Title'] == 'Master')).astype(int)
df['Sex_Pclass'] = df['Sex'] + '_P' + df['Pclass'].astype(str)
df['Age_x_Pclass'] = df['Age'] * df['Pclass']

# 8. Woman-Child Group (WCG) Target Encoding Setup
df['IsWomanOrChild'] = ((df['Sex'] == 'female') | (df['IsChild'] == 1)).astype(int)
df['Group_ID'] = df['Ticket']

print("Engineered DataFrame shape:", df.shape)

### 🔄 Woman-Child Group (WCG) Out-of-Fold Encoding (Strictly No Leakage)

In [ ]:
n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

def calculate_wcg_survival(df_subset, train_indices, test_indices):
    df_res = df_subset.copy()
    df_res['WCG_Survival'] = 0.5
    train_part = df_res.iloc[train_indices]
    
    wcg_groups = train_part[train_part['IsWomanOrChild'] == 1].groupby('Group_ID')['Survived'].agg(['count', 'sum'])
    wcg_valid = wcg_groups[wcg_groups['count'] >= 1]
    
    for idx in range(len(df_res)):
        gid = df_res.iloc[idx]['Group_ID']
        is_wc = df_res.iloc[idx]['IsWomanOrChild']
        
        if gid in wcg_valid.index:
            g_count = wcg_valid.loc[gid, 'count']
            g_sum = wcg_valid.loc[gid, 'sum']
            
            if idx in train_indices and is_wc == 1:
                other_count = g_count - 1
                other_sum = g_sum - df_res.iloc[idx]['Survived']
                if other_count > 0:
                    df_res.iloc[idx, df_res.columns.get_loc('WCG_Survival')] = other_sum / other_count
            else:
                if g_count > 0:
                    df_res.iloc[idx, df_res.columns.get_loc('WCG_Survival')] = g_sum / g_count

    return df_res['WCG_Survival'].values

cat_cols = ['Pclass', 'Sex', 'Embarked', 'Title', 'FamilyType', 'CabinDeck', 'Sex_Pclass', 'IsAlone', 'HasCabin', 'IsChild']
num_cols = ['Age', 'Fare', 'FarePerPerson', 'LogFare', 'LogFarePerPerson', 'SibSp', 'Parch', 'FamilySize', 'TicketGroupSize', 'Age_x_Pclass']

X_base = df[num_cols + cat_cols].copy()
for col in ['Pclass', 'IsAlone', 'HasCabin', 'IsChild']:
    X_base[col] = X_base[col].astype(str)

X_encoded = pd.get_dummies(X_base, columns=cat_cols, drop_first=True)

X_train_raw = X_encoded.iloc[:train_len].copy()
y_train = train_df['Survived'].values
X_test_raw = X_encoded.iloc[train_len:].copy()

# Add Out-Of-Fold WCG Feature
wcg_oof = np.zeros(train_len)
for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train_raw, y_train)):
    wcg_vals = calculate_wcg_survival(df.iloc[:train_len], trn_idx, val_idx)
    wcg_oof[val_idx] = wcg_vals[val_idx]

wcg_full = calculate_wcg_survival(df, list(range(train_len)), list(range(train_len, len(df))))
wcg_test = wcg_full[train_len:]

X_train = X_train_raw.copy()
X_test = X_test_raw.copy()
X_train['WCG_Survival'] = wcg_oof
X_test['WCG_Survival'] = wcg_test

num_cols_all = num_cols + ['WCG_Survival']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols_all] = scaler.fit_transform(X_train[num_cols_all])
X_test_scaled[num_cols_all] = scaler.transform(X_test[num_cols_all])

print(f"X_train matrix shape: {X_train.shape}")
print(f"X_test matrix shape:  {X_test.shape}")


## 4. 🤖 Machine Learning Model Training & 10-Fold CV

In [ ]:
models = {
    'CatBoost': cb.CatBoostClassifier(iterations=200, learning_rate=0.03, depth=4, l2_leaf_reg=3, random_seed=42, verbose=0),
    'LightGBM': lgb.LGBMClassifier(n_estimators=120, learning_rate=0.03, max_depth=3, num_leaves=7, random_state=42, verbose=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=120, learning_rate=0.03, max_depth=3, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss'),
    'RandomForest': RandomForestClassifier(n_estimators=250, max_depth=5, min_samples_split=4, max_features='sqrt', random_state=42),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=250, max_depth=5, min_samples_split=4, max_features='sqrt', random_state=42),
    'HistGradBoost': HistGradientBoostingClassifier(max_iter=120, max_depth=3, l2_regularization=1.0, random_state=42),
    'SVC': SVC(C=1.5, kernel='rbf', probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(C=0.8, max_iter=500, random_state=42)
}

oof_preds = {}
test_preds = {}
cv_results = []

for name, model in models.items():
    oof = np.zeros(len(X_train))
    test_pred_folds = np.zeros(len(X_test))
    
    use_scaled = name in ['SVC', 'LogisticRegression']
    X_tr = X_train_scaled if use_scaled else X_train
    X_te = X_test_scaled if use_scaled else X_test
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_tr, y_train)):
        X_tr_f, y_tr_f = X_tr.iloc[train_idx], y_train[train_idx]
        X_val_f, y_val_f = X_tr.iloc[val_idx], y_train[val_idx]
        
        model.fit(X_tr_f, y_tr_f)
        oof[val_idx] = model.predict_proba(X_val_f)[:, 1]
        test_pred_folds += model.predict_proba(X_te)[:, 1] / n_splits
        
    acc = accuracy_score(y_train, (oof >= 0.5).astype(int))
    auc = roc_auc_score(y_train, oof)
    oof_preds[name] = oof
    test_preds[name] = test_pred_folds
    cv_results.append({'Model': name, 'OOF Accuracy': acc, 'OOF ROC-AUC': auc})

cv_df = pd.DataFrame(cv_results).sort_values(by='OOF Accuracy', ascending=False)
print(cv_df.to_string(index=False))


## 5. 📈 Model Comparison & Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# CV Accuracy Comparison
sns.barplot(data=cv_df, x='OOF Accuracy', y='Model', ax=axes[0], palette='Blues_r')
axes[0].set_title('Out-Of-Fold Accuracy Comparison (10-Fold CV)')
axes[0].set_xlim(0.80, 0.86)

# Feature Importance from Random Forest
rf_model = models['RandomForest']
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(12)
sns.barplot(x=importances.values, y=importances.index, ax=axes[1], palette='viridis')
axes[1].set_title('Top 12 Features Importance (Random Forest)')

plt.tight_layout()
plt.show()


## 6. 🏆 Ensembling & Stacking Meta-Learner

In [ ]:
oof_df = pd.DataFrame(oof_preds)
test_pred_df = pd.DataFrame(test_preds)

top_models = ['CatBoost', 'LightGBM', 'XGBoost', 'RandomForest', 'ExtraTrees', 'SVC', 'HistGradBoost']

# 1. Soft Voting Ensemble
ensemble_oof = oof_df[top_models].mean(axis=1)
ensemble_acc = accuracy_score(y_train, (ensemble_oof >= 0.5).astype(int))
print(f"Top Models Soft Voting Ensemble CV Accuracy: {ensemble_acc:.4f}")

# 2. Meta-Learner Stacking
# Score the meta-learner with its own cross-validation: fitting it on the OOF predictions
# and then scoring it on those same predictions would be an in-sample (optimistic) number.
meta_learner = LogisticRegression(C=0.5, random_state=42)
stacked_oof = cross_val_predict(meta_learner, oof_df[top_models], y_train, cv=skf, method='predict_proba')[:, 1]
stacked_acc = accuracy_score(y_train, (stacked_oof >= 0.5).astype(int))

# Final meta-learner for the test set is fit on all OOF predictions
meta_learner.fit(oof_df[top_models], y_train)

print(f"Stacked Meta-Learner CV Accuracy:            {stacked_acc:.4f}")

# Classification Report
print("\n--- Classification Report (Pure ML Stacked Ensemble) ---")
print(classification_report(y_train, (stacked_oof >= 0.5).astype(int), target_names=['Perished (0)', 'Survived (1)']))

## 7. 🚀 Pure Machine Learning Submission Generation

In [ ]:
# Final predictions using meta-learner stacking on test set
final_test_prob = meta_learner.predict_proba(test_pred_df[top_models])[:, 1]
final_predictions = (final_test_prob >= 0.5).astype(int)

os.makedirs('output', exist_ok=True)
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': final_predictions
})

submission.to_csv('output/submission.csv', index=False)
print("Pure Machine Learning submission file successfully generated at 'output/submission.csv'!")

print("\n--- Submission Sample ---")
print(submission.head(10))

print("\n--- Predicted Class Distribution ---")
print(submission['Survived'].value_counts(normalize=True))
